# Step 4: Wastage Detection
Identify zones where actual consumption significantly exceeds predicted consumption.

In [7]:
import pandas as pd
import joblib
import os
import numpy as np
from pathlib import Path

In [8]:
# Load Model
model_path = os.path.join('..', 'models', 'energy_forecast_model.pkl')
if os.path.exists(model_path):
    model = joblib.load(model_path)
    print("Model loaded successfully.")
else:
    print("Model not found. Please run 03_energy_forecasting.ipynb first.")

# Load Data
data_dir = os.path.join('..', 'data')
try:
    df_energy = pd.read_csv(os.path.join(data_dir, 'energy_readings.csv'))
    df_weather = pd.read_csv(os.path.join(data_dir, 'weather_data.csv'))
    df_occupancy = pd.read_csv(os.path.join(data_dir, 'occupancy_data.csv'))
    df_buildings = pd.read_csv(os.path.join(data_dir, 'buildings.csv'))

    # Merge Data
    df = pd.merge(df_energy, df_occupancy, on=['timestamp', 'building_id'])
    df = pd.merge(df, df_weather, on='timestamp')
    df = pd.merge(df, df_buildings, on='building_id')
    
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Feature Engineering (Must match training)
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Mappings
    occupancy_map = {'Low': 0, 'Medium': 1, 'High': 2}
    df['occupancy_code'] = df['occupancy_level'].map(occupancy_map)
    
    # Recreate building type mapping (dynamic to ensure match if consistent)
    # Ideally should be saved with model, but creating simple map for demo
    building_types = sorted(df['building_type'].unique())
    b_type_map = {b_type: i for i, b_type in enumerate(building_types)}
    df['building_type_code'] = df['building_type'].map(b_type_map)
    
    feature_cols = ['building_type_code', 'area_sq_m', 'temperature', 'humidity', 'occupancy_code', 'hour', 'day_of_week', 'is_weekend']
    
    # Predict
    if 'model' in locals():
        df['predicted_kwh'] = model.predict(df[feature_cols])
        
        # Detect Wastage
        # Logic: If Actual > 1.2 * Predicted -> Wastage
        df['wastage_kwh'] = df['energy_kwh'] - df['predicted_kwh']
        df['is_wastage'] = df['energy_kwh'] > (df['predicted_kwh'] * 1.2)
        
        anomalies = df[df['is_wastage']]
        
        print(f"Total Readings: {len(df)}")
        print(f"Anomalies Detected: {len(anomalies)}")
        
        if len(anomalies) > 0:
            print("\nTop 5 Measurement Anomalies:")
            print(anomalies[['timestamp', 'building_id', 'energy_kwh', 'predicted_kwh', 'wastage_kwh']].head().to_string())
            
            # Save anomalies report
            anomalies.to_csv(os.path.join(data_dir, 'wastage_anomalies.csv'), index=False)
            print(f"\nWastage report saved to {os.path.join(data_dir, 'wastage_anomalies.csv')}")
        else:
            print("No significant wastage detected.")

except FileNotFoundError as e:
    print(f"Data file not found: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

Model loaded successfully.
An error occurred: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- building_type_code
- day_of_week
- humidity
- occupancy_code
Feature names seen at fit time, yet now missing:
- b_type_code
- occ_code

